In [ ]:
import math
from dataclasses import dataclass
from typing import List, Sequence, Tuple

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader

In [ ]:
CONFIG = {
    "lookback": 126,          # paper / repo default total LSTM steps
    "batch_size": 64,
    "num_workers": 0,
    "pin_memory": True,
}

In [ ]:
class _BaseETFWindowDataset(Dataset):
    def __init__(
        self,
        panel: pd.DataFrame,
        feature_cols: Sequence[str],
        endpoint_dates: Sequence[pd.Timestamp],
        lookback: int = 126,
    ):
        super().__init__()
        self.feature_cols = list(feature_cols)
        self.lookback = int(lookback)
        self.endpoint_dates = set(pd.to_datetime(endpoint_dates))

        self.groups = {}
        self.samples: List[Tuple[str, int]] = []

        for ticker, g in panel.groupby("ticker", sort=False):
            g = g.sort_values("date").reset_index(drop=True).copy()

            dates = pd.to_datetime(g["date"]).tolist()
            x = torch.tensor(
                g[self.feature_cols].to_numpy(dtype=np.float32),
                dtype=torch.float32,
            )
            y = torch.tensor(
                g["target_return"].to_numpy(dtype=np.float32),
                dtype=torch.float32,
            )

            asset_id = int(g["asset_id"].iloc[0])

            self.groups[ticker] = {
                "x": x,
                "y": y,
                "dates": dates,
                "asset_id": asset_id,
            }

            for end_idx in range(self.lookback - 1, len(g)):
                end_date = dates[end_idx]
                if end_date in self.endpoint_dates:
                    self.samples.append((ticker, end_idx))

        # keep eval order deterministic / chronological
        self.samples.sort(
            key=lambda item: (
                self.groups[item[0]]["dates"][item[1]],
                item[0],
            )
        )

    def __len__(self) -> int:
        return len(self.samples)

    def _get_common(self, idx: int):
        ticker, end_idx = self.samples[idx]
        g = self.groups[ticker]
        start_idx = end_idx - self.lookback + 1

        x = g["x"][start_idx : end_idx + 1]         # [lookback, num_features]
        y = g["y"][end_idx]                         # scalar target at endpoint
        asset_id = torch.tensor(g["asset_id"], dtype=torch.long)
        end_date = pd.Timestamp(g["dates"][end_idx]).strftime("%Y-%m-%d")

        return x, y, asset_id, end_date, ticker


In [ ]:
class ETFTrainDataset(_BaseETFWindowDataset):
    """
    Train loader only returns what optimization needs.
    """
    def __getitem__(self, idx: int):
        x, y, asset_id, _, _ = self._get_common(idx)
        return x, y, asset_id


class ETFValDataset(_BaseETFWindowDataset):
    def __getitem__(self, idx: int):
        return self._get_common(idx)


class ETFTestDataset(_BaseETFWindowDataset):
    def __getitem__(self, idx: int):
        return self._get_common(idx)

In [ ]:
def train_collate_fn(batch):
    xs, ys, asset_ids = zip(*batch)

    x = torch.stack(xs, dim=0)              # [B, T, F]
    y = torch.stack(ys, dim=0)              # [B]
    asset_ids = torch.stack(asset_ids, dim=0)

    return x, y, asset_ids


def eval_collate_fn(batch):
    xs, ys, asset_ids, dates, tickers = zip(*batch)

    x = torch.stack(xs, dim=0)
    y = torch.stack(ys, dim=0)
    asset_ids = torch.stack(asset_ids, dim=0)

    return x, y, asset_ids, list(dates), list(tickers)


In [ ]:
def build_train_loader(
    panel: pd.DataFrame,
    feature_cols: Sequence[str],
    train_dates: Sequence[pd.Timestamp],
    lookback: int = CONFIG["lookback"],
    batch_size: int = CONFIG["batch_size"],
    num_workers: int = CONFIG["num_workers"],
    pin_memory: bool = CONFIG["pin_memory"],
):
    train_ds = ETFTrainDataset(
        panel=panel,
        feature_cols=feature_cols,
        endpoint_dates=train_dates,
        lookback=lookback,
    )

    train_loader = DataLoader(
        dataset=train_ds,
        batch_size=batch_size,
        shuffle=True,          # train only
        num_workers=num_workers,
        pin_memory=pin_memory,
        collate_fn=train_collate_fn,
    )
    return train_ds, train_loader

In [ ]:
def build_val_loader(
    panel: pd.DataFrame,
    feature_cols: Sequence[str],
    val_dates: Sequence[pd.Timestamp],
    lookback: int = CONFIG["lookback"],
    batch_size: int = CONFIG["batch_size"],
    num_workers: int = CONFIG["num_workers"],
    pin_memory: bool = CONFIG["pin_memory"],
):
    val_ds = ETFValDataset(
        panel=panel,
        feature_cols=feature_cols,
        endpoint_dates=val_dates,
        lookback=lookback,
    )

    val_loader = DataLoader(
        dataset=val_ds,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=pin_memory,
        collate_fn=eval_collate_fn,
    )
    return val_ds, val_loader

In [ ]:
def build_test_loader(
    panel: pd.DataFrame,
    feature_cols: Sequence[str],
    test_dates: Sequence[pd.Timestamp],
    lookback: int = CONFIG["lookback"],
    batch_size: int = CONFIG["batch_size"],
    num_workers: int = CONFIG["num_workers"],
    pin_memory: bool = CONFIG["pin_memory"],
):
    test_ds = ETFTestDataset(
        panel=panel,
        feature_cols=feature_cols,
        endpoint_dates=test_dates,
        lookback=lookback,
    )

    test_loader = DataLoader(
        dataset=test_ds,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=pin_memory,
        collate_fn=eval_collate_fn,
    )
    return test_ds, test_loader